In [ ]:
import re
import requests
from datetime import datetime, timedelta

# ============================================================
# CONFIGURATION
# ============================================================
VISUAL_CROSSING_API_KEY = "V6LAWM6D8QKAEKTKPW2MN4DCV"

# ============================================================
# BACK-END DATA
# ============================================================

def parse_weather_date(text):
    """
    Parse weather date from free text.
    Supports DD-MM-YYYY plus common relative words.
    Returns (date_obj_or_none, error_or_none).
    """
    t = text.lower()

    # Check explicit date first.
    m = re.search(r"\b(\d{2}-\d{2}-\d{4})\b", text)
    if m:
        try:
            return datetime.strptime(m.group(1), "%d-%m-%Y").date(), None
        except ValueError:
            return None, "Invalid date format. Please use DD-MM-YYYY."

    today = datetime.today().date()
    m_rel = re.search(r"\bin\s+(\d{1,2})\s+days?\b", t)
    if m_rel:
        days = int(m_rel.group(1))
        return today + timedelta(days=days), None
    if "day after tomorrow" in t:
        return today + timedelta(days=2), None
    if "tomorrow" in t:
        return today + timedelta(days=1), None
    if "yesterday" in t:
        return today - timedelta(days=1), None
    if "today" in t:
        return today, None

    # Default: today when no date is provided.
    return today, None


def fetch_weather(city, target_date):
    """
    Calls the Visual Crossing Weather API for a specific date.
    Returns a dict with weather info, or a dict with an 'error' key.
    Works for any city in the world.
    """
    date1 = target_date.strftime("%Y-%m-%d")
    date2 = date1
    url = (
        "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/"
        f"{city}/{date1}/{date2}?key={VISUAL_CROSSING_API_KEY}"
    )
    params = {
        "unitGroup": "metric",
        "include": "days",
        "contentType": "json",
    }
    try:
        r = requests.get(url, params=params, timeout=5)
        if r.status_code == 401:
            return {"error": "Invalid API key. Check VISUAL_CROSSING_API_KEY."}
        if r.status_code == 404:
            return {"error": f'City "{city}" not found.'}
        if r.status_code == 400:
            return {
                "error": (
                    f'I could not find a valid city for "{city}". '
                    "Please enter a specific city, for example: 'Weather in Gothenburg'."
                )
            }
        if r.status_code == 429:
            return {"error": "Weather service rate limit reached. Please try again in a moment."}
        if r.status_code >= 500:
            return {"error": "Weather service is temporarily unavailable. Please try again later."}
        r.raise_for_status()
        d = r.json()
        if "days" not in d or not d["days"]:
            return {"error": "No weather data available for the requested date."}

        day = d["days"][0]
        return {
            "temp": round(float(day.get("temp", 0))),
            "feels": round(float(day.get("feelslike", day.get("temp", 0)))),
            "condition": str(day.get("conditions", "Unknown")).capitalize(),
            "wind": f"{round(float(day.get('windspeed', 0)))} km/h",
            "humidity": f"{round(float(day.get('humidity', 0)))}%",
            "date": target_date.strftime("%d-%m-%Y"),
        }
    except requests.exceptions.ConnectionError:
        return {"error": "No internet connection."}
    except requests.exceptions.Timeout:
        return {"error": "Weather request timed out. Please try again."}
    except Exception as e:
        return {"error": "Weather lookup failed due to an unexpected error. Please try a different city."}


# Hardcoded restaurant database
RESTAURANT_DB = [
    {"name": "Smaka",           "cuisine": "swedish", "rating": 4.6, "area": "central",    "price": "$$",  "hours": "11:00-22:00"},
    {"name": "Rakultur",        "cuisine": "japanese",   "rating": 4.8, "area": "central",    "price": "$$$", "hours": "12:00-22:00"},
    {"name": "Mr Pizza",        "cuisine": "italian",   "rating": 4.2, "area": "hisingen",   "price": "$",   "hours": "10:00-23:00"},
    {"name": "Dhaba",           "cuisine": "indian",  "rating": 4.5, "area": "lindholmen", "price": "$$",  "hours": "11:30-21:30"},
    {"name": "Tacos & Tequila", "cuisine": "mexican", "rating": 4.3, "area": "central",    "price": "$$",  "hours": "12:00-00:00"},
    {"name": "Nori",            "cuisine": "japanese",   "rating": 4.4, "area": "hisingen",   "price": "$$",  "hours": "11:00-21:00"},
    {"name": "Punjab Palace",   "cuisine": "indian",  "rating": 4.3, "area": "central",    "price": "$$",  "hours": "12:00-22:00"},
]

# Hardcoded tram/bus schedule
TRANSPORT_DB = {
    "1":  [{"dest": "Angered",          "dep": "+4 min",  "platform": "A"},
           {"dest": "Angered",          "dep": "+19 min", "platform": "A"}],
    "2":  [{"dest": "Molndal",          "dep": "+2 min",  "platform": "B"},
           {"dest": "Molndal",          "dep": "+17 min", "platform": "B"}],
    "6":  [{"dest": "Lansmansgarden",   "dep": "+8 min",  "platform": "C"},
           {"dest": "Lansmansgarden",   "dep": "+23 min", "platform": "C"}],
    "11": [{"dest": "Saltholmen",       "dep": "+6 min",  "platform": "D"},
           {"dest": "Saltholmen",       "dep": "+21 min", "platform": "D"}],
    "16": [{"dest": "Hoegsbo",          "dep": "+3 min",  "platform": "A"},
           {"dest": "Hoegsbo",          "dep": "+18 min", "platform": "A"}],
    "55": [{"dest": "Angered C.",       "dep": "+1 min",  "platform": "E"},
           {"dest": "Angered C.",       "dep": "+16 min", "platform": "E"}],
}

# Hardcoded directions between known locations
DIRECTIONS_DB = {
    ("central station", "liseberg"):             {"duration": "12 min", "distance": "3.2 km", "mode": "tram 5 or 6"},
    ("central station", "lindholmen"):           {"duration": "8 min",  "distance": "2.1 km", "mode": "ferry or bus 16"},
    ("central station", "hisingen"):             {"duration": "15 min", "distance": "4.5 km", "mode": "bus 16 or 32"},
    ("central station", "gothenburg university"):{"duration": "20 min", "distance": "5.1 km", "mode": "tram 13"},
    ("liseberg", "central station"):             {"duration": "12 min", "distance": "3.2 km", "mode": "tram 5 or 6"},
    ("lindholmen", "central station"):           {"duration": "8 min",  "distance": "2.1 km", "mode": "ferry or bus 16"},
}

def lookup_directions(origin, destination):
    key = (origin.lower().strip(), destination.lower().strip())
    return DIRECTIONS_DB.get(key) or DIRECTIONS_DB.get((key[1], key[0]))

# ============================================================
# CONVERSATION CONTEXT (slot store)
# ============================================================
class ConversationContext:
    """
    Persists named slots across turns, e.g. city='london', cuisine='sushi'.
    Also tracks the last matched intent for follow-up resolution.

    Example
    -------
    Bot:  "Which city?"
    User: "London"          <- no intent match, but last_intent='weather'
                               so DialogueManager re-runs weather with 'London'
    """
    def __init__(self):
        self.slots       = {}
        self.last_intent = None
        self.turn        = 0

    def set(self, key, value):  self.slots[key] = value
    def get(self, key):         return self.slots.get(key)
    def clear(self, key):       self.slots.pop(key, None)
    def clear_all(self):        self.slots.clear()

    def remember(self, intent_name):
        self.last_intent = intent_name
        self.turn += 1

    def __repr__(self):
        return f"Context(turn={self.turn}, last={self.last_intent}, slots={self.slots})"


# ============================================================
# INTENT & INTENT REGISTRY
# ============================================================
class Intent:
    """
    One capability of the assistant.

    Parameters
    ----------
    name       : str
    patterns   : list of str (substring) or compiled regex objects
    extract_fn : (text, ctx) -> None   pulls slot values from user input
    respond_fn : (ctx)       -> str    builds reply from current slots
    """
    def __init__(self, name, patterns, extract_fn, respond_fn):
        self.name       = name
        self.patterns   = patterns
        self.extract    = extract_fn
        self.respond    = respond_fn

    def matches(self, text):
        t = text.lower()
        for p in self.patterns:
            if isinstance(p, str):
                # Use token-aware matching for literal phrases so
                # short strings like "hi" do not match inside words
                # such as "hisingen".
                if re.search(rf"\b{re.escape(p.lower())}\b", t):
                    return True
            if hasattr(p, "search") and p.search(t):
                return True
        return False


class IntentRegistry:
    """
    Holds all intents. Add a new skill via register() — nothing else changes.
    Matching is a linear scan; first match wins.
    """
    def __init__(self):          self._intents = []
    def register(self, intent):  self._intents.append(intent)
    def names(self):             return [i.name for i in self._intents]

    def match(self, text):
        for i in self._intents:
            if i.matches(text): return i
        return None

    def get_by_name(self, name):
        for i in self._intents:
            if i.name == name: return i
        return None


# ============================================================
# INTENT DEFINITIONS
# ============================================================
registry = IntentRegistry()

# ── GREETING ──────────────────────────────────────────────────
registry.register(Intent(
    name="greeting",
    patterns=["hello", "hi", "hey", "good morning", "good evening", "howdy"],
    extract_fn=lambda t, c: None,
    respond_fn=lambda c: (
        "Hello! I can help you with:\n"
        "  - Weather forecasts (live data)\n"
        "  - Restaurant suggestions\n"
        "  - Tram/bus departure times\n"
        "  - Directions between places\n"
        "Type 'help' for example questions."
    ),
))

# ── WEATHER ───────────────────────────────────────────────────
def weather_extract(text, ctx):
    parsed_date, date_error = parse_weather_date(text)
    if date_error:
        ctx.set("date_error", date_error)
    else:
        ctx.set("date", parsed_date)
        ctx.clear("date_error")

    # Remove date tokens/keywords before extracting city so
    # inputs like "weather in gothenburg 11-03-2026" are handled.
    city_source = re.sub(r"\b\d{2}-\d{2}-\d{4}\b", "", text, flags=re.IGNORECASE)
    city_source = re.sub(r"\bin\s+\d{1,2}\s+days?\b", "", city_source, flags=re.IGNORECASE)
    city_source = re.sub(
        r"\b(today|tomorrow|yesterday|day after tomorrow)\b",
        "",
        city_source,
        flags=re.IGNORECASE,
    )

    m = re.search(r"\b(?:in|for|at)\b\s+([a-zA-Z\s]+?)(?:\?|$|,)", city_source, re.IGNORECASE)
    if m:
        candidate = m.group(1).strip()
        if candidate and not re.search(r"\b(weather|forecast|temperature)\b", candidate, re.IGNORECASE):
            ctx.set("city", candidate)
        return

    # If this is a generic weather question with no location, force slot filling
    # instead of reusing a previously remembered city.
    if re.search(r"\b(weather|forecast|temperature)\b", text, re.IGNORECASE):
        ctx.clear("city")
        return

    # Follow-up support: accept plain city names like "London".
    if re.search(r"\b(weather|forecast|temperature|raining|sunny|cold|hot|today|tomorrow|yesterday)\b", text, re.IGNORECASE):
        return
    cleaned = text.strip().strip("?.!,")
    if re.fullmatch(r"[A-Za-z][A-Za-z\s\-']{0,49}", cleaned):
        ctx.set("city", cleaned)

def weather_respond(ctx):
    date_error = ctx.get("date_error")
    if date_error:
        return date_error

    city = ctx.get("city")
    if not city:
        return "Which city would you like the weather for?"
    target_date = ctx.get("date") or datetime.today().date()
    result = fetch_weather(city, target_date)
    if "error" in result:
        ctx.clear("city")
        return f"Weather lookup failed: {result['error']}"
    return (
        f"Weather in {city.title()} on {result['date']}:\n"
        f"  Condition  : {result['condition']}\n"
        f"  Temp       : {result['temp']}C (feels like {result['feels']}C)\n"
        f"  Wind       : {result['wind']}\n"
        f"  Humidity   : {result['humidity']}"
    )

registry.register(Intent(
    name="weather",
    patterns=[re.compile(r"weather|forecast|temperature|raining|sunny|cold outside|hot outside")],
    extract_fn=weather_extract,
    respond_fn=weather_respond,
))

# ── RESTAURANT ────────────────────────────────────────────────
CUISINES = ["japanese", "italian", "indian", "swedish", "mexican"]
# Map common user terms to cuisines supported by the demo data.
CUISINE_SYNONYMS = {
    "sushi": "japanese",
    "pizza": "italian",
    "mexico": "mexican",
    "india": "indian",
    "sweden": "swedish",
}
AREAS    = ["central", "hisingen", "lindholmen"]

def restaurant_extract(text, ctx):
    t = text.lower()
    found_cuisine = False
    found_area = False
    ctx.clear("invalid_cuisine")
    ctx.clear("invalid_area")

    for alias, canonical in CUISINE_SYNONYMS.items():
        if re.search(rf"\b{re.escape(alias)}\b", t):
            ctx.set("cuisine", canonical)
            found_cuisine = True
    for c in CUISINES:
        if re.search(rf"\b{re.escape(c)}\b", t):
            ctx.set("cuisine", c)
            found_cuisine = True
    for a in AREAS:
        if re.search(rf"\b{re.escape(a)}\b", t):
            ctx.set("area", a)
            found_area = True

    # If user specifies "<word> food" or "eat <word>" and it is unknown,
    # return a helpful available-cuisines message instead of generic output.
    cuisine_guess = None
    m_food = re.search(r"\b([a-z]+)\s+food\b", t)
    if m_food:
        cuisine_guess = m_food.group(1)
    else:
        m_eat = re.search(r"\beat\s+([a-z]+)\b", t)
        if m_eat:
            cuisine_guess = m_eat.group(1)

    known_cuisine_terms = set(CUISINES) | set(CUISINE_SYNONYMS.keys())
    if cuisine_guess and cuisine_guess not in known_cuisine_terms:
        ctx.set("invalid_cuisine", cuisine_guess)

    # If user says "in <area>" and area is unknown, provide valid options.
    m_area = re.search(r"\bin\s+([a-z\s]+?)(?:\?|$|,)", t)
    if m_area and not found_area:
        area_guess = m_area.group(1).strip()
        if area_guess and area_guess not in AREAS:
            ctx.set("invalid_area", area_guess)

    generic_query = bool(re.search(r"\b(top restaurants?|restaurants?|food|eat|hungry|dinner|lunch)\b", t))
    if generic_query and not found_cuisine:
        ctx.clear("cuisine")
    if generic_query and not found_area:
        ctx.clear("area")

def restaurant_respond(ctx):
    invalid_cuisine = ctx.get("invalid_cuisine")
    if invalid_cuisine:
        ctx.clear("invalid_cuisine")
        available_cuisines = ", ".join(sorted(CUISINES))
        return (
            f'I do not have cuisine "{invalid_cuisine}" in this demo. '
            f"Available cuisines are: {available_cuisines}."
        )

    invalid_area = ctx.get("invalid_area")
    if invalid_area:
        ctx.clear("invalid_area")
        available_areas = ", ".join(AREAS)
        return (
            f'I do not have area "{invalid_area}" in this demo. '
            f"Available areas are: {available_areas}."
        )

    results = list(RESTAURANT_DB)
    cuisine = ctx.get("cuisine")
    area    = ctx.get("area")
    if cuisine: results = [r for r in results if r["cuisine"] == cuisine]
    if area:    results = [r for r in results if r["area"]    == area]
    if not results:
        return "No restaurants found — try a different cuisine or area."
    top   = sorted(results, key=lambda r: r["rating"], reverse=True)[:3]
    label = f" ({cuisine})" if cuisine else ""
    lines = [f"Top restaurants{label}:"]
    for r in top:
        lines.append(
            f"  {r['name']:20s}  {r['cuisine'].title():8s}  "
            f"Rating: {r['rating']}  {r['price']}  Open: {r['hours']}"
        )
    return "\n".join(lines)

registry.register(Intent(
    name="restaurant",
    patterns=[re.compile(r"restaurant|eat|food|hungry|dinner|lunch|italian|japanese|indian|swedish|mexican")],
    extract_fn=restaurant_extract,
    respond_fn=restaurant_respond,
))

# ── TRANSPORT ─────────────────────────────────────────────────
def transport_extract(text, ctx):
    t = text.lower()
    found_line = False
    ctx.clear("invalid_line")
    m = re.search(r"\b(\d{1,2})\b", text)
    if m and m.group(1) in TRANSPORT_DB:
        ctx.set("line", m.group(1))
        found_line = True
    elif m:
        ctx.set("invalid_line", m.group(1))

    # Generic transport requests should not reuse stale line values.
    generic_query = bool(re.search(r"\b(tram|bus|line|next|departure|arrive|transit)\b", t))
    if generic_query and not found_line:
        ctx.clear("line")

def transport_respond(ctx):
    invalid_line = ctx.get("invalid_line")
    if invalid_line:
        ctx.clear("invalid_line")
        available = ", ".join(sorted(TRANSPORT_DB.keys(), key=int))
        return f'I do not have line {invalid_line}. Available lines are: {available}'

    line = ctx.get("line")
    if not line:
        available = ", ".join(sorted(TRANSPORT_DB.keys(), key=int))
        return f"Which line? I have schedules for: {available}"
    deps  = TRANSPORT_DB[line]
    lines = [f"Line {line} upcoming departures:"]
    for d in deps:
        lines.append(f"  Towards {d['dest']:20s}  {d['dep']:8s}  Platform {d['platform']}")
    return "\n".join(lines)

registry.register(Intent(
    name="transport",
    patterns=[re.compile(r"tram|bus|line \d+|next.*\d+|departure|when.*arrive|transit")],
    extract_fn=transport_extract,
    respond_fn=transport_respond,
))

# ── DIRECTIONS ────────────────────────────────────────────────
def directions_extract(text, ctx):
    t = text.lower()
    generic_direction_request = bool(re.search(r"\b(direction|directions|route|navigate|get to|go to|help)\b", t))

    # Explicit "directions from X" form: set origin and ask destination.
    m_from_only = re.search(r"^(?:directions?\s+)?from\s+(.+?)(?:\?|$)", t)
    if m_from_only and " to " not in t:
        ctx.set("origin", m_from_only.group(1).strip())
        ctx.clear("destination")
        ctx.set("awaiting_direction_slot", "destination")
        return

    # Compact pair form: "<origin> to <destination>".
    # This supports replies like "central station to hisingen".
    compact_pair = re.search(r"^\s*(.+?)\s+to\s+(.+?)\s*(?:\?|$)", t)
    if (
        compact_pair
        and not t.startswith("to ")
        and not re.match(r"^\s*(direction|directions|route|navigate|get|go|help)\b", t)
    ):
        ctx.set("origin", compact_pair.group(1).strip())
        ctx.set("destination", compact_pair.group(2).strip())
        ctx.clear("awaiting_direction_slot")
        return

    # If we explicitly asked for a missing slot, treat plain input as that slot.
    awaiting = ctx.get("awaiting_direction_slot")
    if awaiting in ("origin", "destination"):
        cleaned = t.strip().strip("?.!,")
        if cleaned and re.fullmatch(r"[a-z0-9][a-z0-9\s\-\.']{0,79}", cleaned):
            ctx.set(awaiting, cleaned)
            ctx.clear("awaiting_direction_slot")
            return

    m = re.search(r"from\s+(.+?)\s+to\s+(.+?)(?:\?|$)", t)
    if m:
        ctx.set("origin",      m.group(1).strip())
        ctx.set("destination", m.group(2).strip())
        ctx.clear("awaiting_direction_slot")
        return
    m = re.search(r"to\s+(.+?)\s+from\s+(.+?)(?:\?|$)", t)
    if m:
        ctx.set("destination", m.group(1).strip())
        ctx.set("origin",      m.group(2).strip())
        ctx.clear("awaiting_direction_slot")
        return
    m = re.search(r"(?:directions?|get|go|navigate)\s+to\s+(.+?)(?:\?|$)", t)
    if m:
        # New destination-only request: clear stale origin from previous turns.
        ctx.set("destination", m.group(1).strip())
        ctx.clear("origin")
        ctx.clear("awaiting_direction_slot")
        return

    # Follow-up shorthand after "Where would you like directions to?"
    if (
        re.fullmatch(r"[a-z0-9][a-z0-9\s\-\.']{0,79}", t.strip().strip("?.!,"))
        and not ctx.get("destination")
        and not generic_direction_request
    ):
        ctx.set("destination", t.strip().strip("?.!,"))
        ctx.clear("awaiting_direction_slot")
        return

    # Generic directions request (e.g., "direction help") should start a fresh frame
    # instead of reusing stale slots from previous route queries.
    if generic_direction_request:
        ctx.clear("origin")
        ctx.clear("destination")
        ctx.set("awaiting_direction_slot", "destination")

def directions_respond(ctx):
    dest   = ctx.get("destination")
    origin = ctx.get("origin")
    if not dest:
        ctx.set("awaiting_direction_slot", "destination")
        return "Where would you like directions to?"
    if not origin:
        ctx.set("awaiting_direction_slot", "origin")
        return "Where are you starting from?"
    ctx.clear("awaiting_direction_slot")
    d = lookup_directions(origin, dest)
    if not d:
        return (
            f'Sorry, I don\'t have directions from "{origin}" to "{dest}".\n'
            "Known places: Central Station, Liseberg, Lindholmen, Hisingen, Gothenburg University."
        )
    return (
        f"Directions from {origin.title()} to {dest.title()}:\n"
        f"  Duration : {d['duration']}\n"
        f"  Distance : {d['distance']}\n"
        f"  Take     : {d['mode']}"
    )

registry.register(Intent(
    name="directions",
    patterns=[
        re.compile(r"direction|how (do i|to) get|route to|navigate|get to|go to"),
        re.compile(r"^\s*[a-z0-9\s\-\.']+\s+to\s+[a-z0-9\s\-\.']+\s*$"),
        re.compile(r"^\s*(?:directions?\s+)?from\s+.+$"),
    ],
    extract_fn=directions_extract,
    respond_fn=directions_respond,
))

# ── HELP ──────────────────────────────────────────────────────
registry.register(Intent(
    name="help",
    patterns=["help", "what can you do", "capabilities", "options"],
    extract_fn=lambda t, c: None,
    respond_fn=lambda c: (
        "Here is what I can do:\n"
        "  Weather    : 'What is the weather in London?'\n"
        "  Restaurant : 'Find me a sushi restaurant'\n"
        "             : 'Any indian food in Hisingen?'\n"
        "  Transport  : 'When is the next tram on line 16?'\n"
        "  Directions : 'Directions from Central Station to Liseberg'\n"
        "You can ask in two steps — I remember what you said!"
    ),
))

# ── THANKS ────────────────────────────────────────────────────
registry.register(Intent(
    name="thanks",
    patterns=["thank", "thanks", "cheers", "great", "perfect", "awesome"],
    extract_fn=lambda t, c: None,
    respond_fn=lambda c: "You are welcome! Anything else?",
))


# ============================================================
# DIALOGUE MANAGER
# ============================================================
class DialogueManager:
    """
    Processes one user turn at a time.

    Flow
    ----
    1. Match an intent from the registry.
    2. If matched  -> extract slots, generate response, save intent name.
    3. If no match -> follow-up resolution: re-run the last intent's
       extract + respond with the new input. Handles cases like:
         Bot: "Which city?"  ->  User: "London"
         Bot: "Which line?"  ->  User: "16"
    4. If still nothing -> generic fallback.
    """
    def __init__(self, registry):
        self.registry = registry
        self.ctx      = ConversationContext()

    def _expects_follow_up(self):
        """
        Returns True only when the assistant is explicitly waiting for
        missing slot information from the user.
        """
        if self.ctx.last_intent == "weather":
            return not self.ctx.get("city")
        if self.ctx.last_intent == "transport":
            return not self.ctx.get("line")
        if self.ctx.last_intent == "directions":
            return bool(self.ctx.get("awaiting_direction_slot")) or not self.ctx.get("destination") or not self.ctx.get("origin")
        return False

    def process(self, text):
        text = text.strip()
        normalized_text = text.lower()
        intent = self.registry.match(normalized_text)

        if intent:
            intent.extract(normalized_text, self.ctx)
            reply = intent.respond(self.ctx)
            self.ctx.remember(intent.name)
            return reply

        # Follow-up: user is likely answering a clarification question
        if self.ctx.last_intent and self._expects_follow_up():
            last = self.registry.get_by_name(self.ctx.last_intent)
            if last:
                last.extract(normalized_text, self.ctx)
                reply = last.respond(self.ctx)
                self.ctx.remember(last.name)
                return reply

        return (
            "I didn't quite get that. "
            "Try asking about weather, restaurants, transport, or directions. "
            "Type 'help' for examples."
        )

    def reset(self):
        self.ctx = ConversationContext()


# ============================================================
# MAIN CHAT LOOP
# ============================================================
def run():
    dm = DialogueManager(registry)
    print("=" * 60)
    print("  Digital Assistant  (quit to exit | reset to clear memory)")
    print("=" * 60)
    print(f"Assistant: {dm.process('hello')}\n")

    while True:
        try:
            user = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n[Session ended]")
            break

        if not user:
            continue
        if user.lower() in ("quit", "exit"):
            print("Assistant: Goodbye!")
            break
        if user.lower() == "reset":
            dm.reset()
            print("Assistant: Memory cleared. Starting fresh!\n")
            continue

        print(f"Assistant: {dm.process(user)}\n")


if __name__ == "__main__":
    run()
